In [1]:
"""
EcoGuide AI - Local Gradio App
Runs entirely on your laptop with GPU. No APIs needed.
"""

import json
import re
import faiss
import numpy as np
import torch
import gradio as gr
from pathlib import Path
from typing import List, Dict
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer as LLMTokenizer
from sentence_transformers import SentenceTransformer


# ═════════════════════════════════════════════════════
# 1. GUARDRAILS (same as your notebook)
# ═════════════════════════════════════════════════════
class Guardrails:
    CONFIDENCE_THRESHOLD = 0.6
    MARGIN_THRESHOLD = 0.1
    MIN_QUERY_WORDS = 2
    MAX_QUERY_CHARS = 500
    MIN_RELEVANCE_SCORE = 0.6

    CLARIFICATION = {
        "en": (
            "I'm not sure I understood your question. "
            "Could you rephrase or specify whether you're asking about "
            "Premium Sortify's features or general recycling information?"
        ),
        "ar": (
            "لست متأكداً من فهم سؤالك. "
            "هل يمكنك إعادة صياغته أو تحديد ما إذا كنت تسأل عن "
            "ميزات بريميم سورتيفاي أو معلومات عامة حول إعادة التدوير؟"
        ),
    }

    FALLBACK = {
        "en": (
            "I don't have specific information about that in my knowledge base. "
            "Can I help you with something else about Premium Sortify or recycling?"
        ),
        "ar": (
            "ليس لدي معلومات محددة عن ذلك في قاعدة معرفتي. "
            "هل يمكنني مساعدتك في شيء آخر يخص بريميم سورتيفاي أو إعادة التدوير؟"
        ),
    }

    NEGATION_RE = re.compile(
        r"\b(don't|do not|doesn\'t|didn\'t|won\'t|wouldn\'t|shouldn\'t|can\'t|cannot|never|no)\b|"
        r"\b(لا|لست|لم|لن|غير|بدون|من غير|مش|مو|ليس|ليست)\b",
        re.IGNORECASE,
    )

    @staticmethod
    def detect_language(text: str) -> str:
        arabic_chars = sum(1 for c in text if "\u0600" <= c <= "\u06FF")
        return "ar" if arabic_chars / max(len(text), 1) > 0.30 else "en"

    @staticmethod
    def is_too_short(text: str) -> bool:
        return len(text.strip().split()) < Guardrails.MIN_QUERY_WORDS

    @staticmethod
    def is_too_long(text: str) -> bool:
        return len(text) > Guardrails.MAX_QUERY_CHARS

    @staticmethod
    def has_negation(text: str) -> bool:
        return bool(Guardrails.NEGATION_RE.search(text))

    def pre_rag_check(self, query: str, intent: str, confidence: float, margin: float):
        if intent == "out_of_scope":
            lang = self.detect_language(query)
            return False, self.CLARIFICATION[lang], "out_of_scope"
        if confidence < self.CONFIDENCE_THRESHOLD:
            lang = self.detect_language(query)
            return False, self.CLARIFICATION[lang], "low_confidence"
        if margin < self.MARGIN_THRESHOLD:
            lang = self.detect_language(query)
            return False, self.CLARIFICATION[lang], "low_margin"
        if self.is_too_short(query):
            lang = self.detect_language(query)
            return False, self.CLARIFICATION[lang], "too_short"
        if self.is_too_long(query):
            lang = self.detect_language(query)
            return False, self.CLARIFICATION[lang], "too_long"
        if self.has_negation(query):
            return True, None, "negation_detected"
        return True, None, "passed"

    def post_rag_check(self, retrieved_chunks: list):
        if not retrieved_chunks:
            return False, self.FALLBACK["en"]
        top_score = retrieved_chunks[0].get("score", 0.0)
        if top_score < self.MIN_RELEVANCE_SCORE:
            return False, self.FALLBACK["en"]
        return True, None



In [2]:

# ═════════════════════════════════════════════════════
# 2. INTENT CLASSIFIER
# ═════════════════════════════════════════════════════
class EcoGuideIntentClassifier:
    ID2LABEL = {
        0: "out_of_scope",
        1: "project_information",
        2: "recycle_glass",
        3: "recycle_metal",
        4: "recycle_organic",
        5: "recycle_plastic",
        6: "sustainability_definition",
        7: "waste_sorting",
    }

    def __init__(self, model_path: str = "best intent classifier"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_path = Path(model_path)

        if not self.model_path.exists():
            raise FileNotFoundError(f"Model not found at: {self.model_path.absolute()}")

        print(f"[Classifier] Loading from {self.model_path} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            str(self.model_path), trust_remote_code=True
        )
        self.model = AutoModelForSequenceClassification.from_pretrained(
            str(self.model_path), trust_remote_code=True
        ).to(self.device)
        self.model.eval()

        num_labels = self.model.config.num_labels
        assert num_labels == len(self.ID2LABEL)
        print(f"[Classifier] Ready on {self.device} | {num_labels} labels")

    @torch.inference_mode()
    def classify(self, text: str) -> dict:
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        ).to(self.device)

        outputs = self.model(**inputs)
        logits = outputs.logits[0]
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

        top2_idx = np.argsort(probs)[-2:][::-1]
        top1_idx = int(top2_idx[0])
        top2_idx_val = int(top2_idx[1])

        confidence = float(probs[top1_idx])
        margin = float(probs[top1_idx] - probs[top2_idx_val])
        intent = self.ID2LABEL.get(top1_idx, "out_of_scope")

        all_scores = {
            self.ID2LABEL.get(i, f"label_{i}"): float(probs[i])
            for i in range(len(probs))
        }

        return {
            "intent": intent,
            "confidence": confidence,
            "margin": margin,
            "all_scores": all_scores,
        }


# ═════════════════════════════════════════════════════
# 3. CONFIG
# ═════════════════════════════════════════════════════
class EcoGuideConfig:
    llm_model: str = "Qwen/Qwen2.5-7B-Instruct"
    llm_dtype_str: str = "bfloat16"
    llm_max_new_tokens: int = 1024
    llm_temperature: float = 0.3
    llm_top_p: float = 0.9
    llm_context_tokens: int = 3500

    project_index_path: Path = Path("ecoguide/data/project_index.faiss")
    project_chunks_path: Path = Path("ecoguide/data/project_index.jsonl")
    general_index_path: Path = Path("ecoguide/data/general_index.faiss")
    general_chunks_path: Path = Path("ecoguide/data/general_index.jsonl")

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    @property
    def llm_dtype(self) -> torch.dtype:
        return {
            "bfloat16": torch.bfloat16,
            "float16": torch.float16,
            "float32": torch.float32,
        }.get(self.llm_dtype_str, torch.bfloat16)


CONFIG = EcoGuideConfig()


# ═════════════════════════════════════════════════════
# 4. RETRIEVER
# ═════════════════════════════════════════════════════
class FaissIndex:
    def __init__(self, faiss_path: Path, chunks_path: Path, name: str):
        self.name = name
        print(f"[Retriever:{name}] Loading from {faiss_path} ...")
        self.index = faiss.read_index(str(faiss_path))

        with open(chunks_path, "r", encoding="utf-8") as f:
            self.chunks = [json.loads(line) for line in f]

        assert self.index.ntotal == len(self.chunks)
        print(f"[Retriever:{name}] Ready: {self.index.ntotal} vectors")

    def search(self, query_vec: np.ndarray, top_k: int = 15):
        query_vec = query_vec.astype("float32").reshape(1, -1)
        scores, ids = self.index.search(query_vec, top_k)

        results = []
        for score, idx in zip(scores[0], ids[0]):
            if idx == -1:
                continue
            result = self.chunks[idx].copy()
            result["score"] = float(score)
            result["id"] = result.get("id", f"{self.name}_{idx}")
            results.append(result)
        return results


# ═════════════════════════════════════════════════════
# 5. LLM GENERATOR
# ═════════════════════════════════════════════════════
def detect_language(text: str) -> str:
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    return "ar" if arabic_chars / max(len(text), 1) > 0.3 else "en"


class LLMGenerator:
    SYSTEM_PROMPT = (
        "You are EcoGuide AI, the sustainability assistant for Premium Sortify "
        "at Cairo University Faculty of Engineering."
        "Respond ONLY in the same language as the user's query (English or Arabic). Do not use any other language"
    )

    def __init__(self):
        self.device = CONFIG.device
        self.model_name = CONFIG.llm_model
        self._model = None
        self._tokenizer = None

    def _load(self):
        if self._model is not None:
            return

        print(f"[Generator] Loading {self.model_name} ...")
        self._tokenizer = LLMTokenizer.from_pretrained(
            self.model_name, trust_remote_code=True, padding_side="left"
        )
        if self._tokenizer.pad_token is None:
            self._tokenizer.pad_token = self._tokenizer.eos_token

        load_kwargs = {
            "torch_dtype": CONFIG.llm_dtype,
            "device_map": "auto",
            "trust_remote_code": True,
            "attn_implementation": "sdpa",
        }

        self._model = AutoModelForCausalLM.from_pretrained(self.model_name, **load_kwargs)
        self._model.eval()
        print("[Generator] Ready.")

    def _build_prompt(self, question: str, context_chunks: List[Dict]) -> str:
        lang = detect_language(question)

        if lang == "ar":
            lang_instruction = (
                "يجب أن تجيب باللغة العربية فقط. لا تستخدم اي لغة اخري أبداً. "
                "استخدم السياق المقدم للإجابة على السؤال التالي:"
            )
        else:
            lang_instruction = (
                "You MUST answer in English only. Do not use any other language. "
                "Use the provided context to answer the following question:"
            )

        context_parts = []
        current_tokens = 0
        for chunk in context_chunks:
            text = chunk["text"]
            source = chunk.get("source", "Unknown")
            heading = chunk.get("heading_path", "")
            entry = f"[Source: {source} | {heading}]\n{text}\n"
            entry_tokens = len(self._tokenizer.encode(entry))
            if current_tokens + entry_tokens > CONFIG.llm_context_tokens:
                break
            context_parts.append(entry)
            current_tokens += entry_tokens

        context_block = "\n---\n".join(context_parts)

        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"{lang_instruction}\n\n"
                f"Context:\n{context_block}\n\n"
                f"Question: {question}\n\n"
                f"Answer:"
            )}
        ]

        return self._tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    @torch.inference_mode()
    def generate(self, question: str, context_chunks: List[Dict]) -> Dict:
        self._load()
        prompt = self._build_prompt(question, context_chunks)

        inputs = self._tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096,
        ).to(self.device)

        outputs = self._model.generate(
            **inputs,
            max_new_tokens=CONFIG.llm_max_new_tokens,
            temperature=CONFIG.llm_temperature,
            top_p=CONFIG.llm_top_p,
            do_sample=True,
            pad_token_id=self._tokenizer.pad_token_id,
            eos_token_id=self._tokenizer.eos_token_id,
        )

        generated = outputs[0][inputs.input_ids.shape[1]:]
        answer = self._tokenizer.decode(generated, skip_special_tokens=True).strip()

        sources = list({f"{c.get('source','')} | {c.get('heading_path','')}"
                        for c in context_chunks})

        return {
            "answer": answer,
            "sources": sources,
            "model": self.model_name,
            "language": detect_language(question),
        }


# ═════════════════════════════════════════════════════
# 6. RAG PIPELINE
# ═════════════════════════════════════════════════════
ROUTING_MAP = {
    "project_information":       ["project"],
    "waste_sorting":             ["project"],
    "recycle_plastic":           ["project", "general"],
    "recycle_glass":             ["project", "general"],
    "recycle_metal":             ["project", "general"],
    "recycle_organic":           ["project", "general"],
    "sustainability_definition": ["general"],
    "out_of_scope":              [],
}


class EcoGuideRAG:
    def __init__(self):
        print("[RAG] Initializing...")
        self.guardrails = Guardrails()
        self.classifier = EcoGuideIntentClassifier("best intent classifier")
        self.generator = LLMGenerator()
        self.embedder = SentenceTransformer(
            "intfloat/multilingual-e5-base",
            device=CONFIG.device,
            trust_remote_code=True
        )

        self.project_idx = FaissIndex(
            CONFIG.project_index_path,
            CONFIG.project_chunks_path,
            "project"
        )
        self.general_idx = FaissIndex(
            CONFIG.general_index_path,
            CONFIG.general_chunks_path,
            "general"
        )
        print("[RAG] Ready.")

    def _embed_query(self, text: str) -> np.ndarray:
        return self.embedder.encode(f"query: {text}", normalize_embeddings=True)

    def _fuse_rrf(self, results_per_index: dict, k: int = 60) -> list:
        scores = {}
        for index_name, results in results_per_index.items():
            for rank, result in enumerate(results):
                doc_id = result.get("id", f"{index_name}_{rank}")
                if doc_id not in scores:
                    scores[doc_id] = {"chunk": result, "rrf_score": 0.0}
                scores[doc_id]["rrf_score"] += 1.0 / (k + rank + 1)

        fused = sorted(scores.values(), key=lambda x: x["rrf_score"], reverse=True)
        return [item["chunk"] for item in fused[:5]]

    def answer(self, query: str) -> dict:
        # ── Keyword override for project name ──────────────────────
        lower = query.lower()
        if "premium sortify" in lower or "بريميم" in lower or "سورتيفاي" in lower or "sortify" in lower or "Premium Sortify" in query or "Sortify" in query:
            classification = {
                "intent": "project_information",
                "confidence": 0.999,
                "margin": 0.999,
                "all_scores": {}
            }
            intent = "project_information"
            confidence = 0.999
            margin = 0.999
            print(f"[RAG] Keyword override: project_information (Premium Sortify detected)")
        else:
            classification = self.classifier.classify(query)
            intent = classification["intent"]
            confidence = classification["confidence"]
            margin = classification["margin"]
        # ── End override ───────────────────────────────────────────
        
        print(f"[RAG] {intent} | conf={confidence:.3f} | margin={margin:.3f}")
        # ... rest of the method stays exactly the same
        classification = self.classifier.classify(query)
        intent = classification["intent"]
        confidence = classification["confidence"]
        margin = classification["margin"]
        print(f"[RAG] {intent} | conf={confidence:.3f} | margin={margin:.3f}")

        passed, clarification, reason = self.guardrails.pre_rag_check(
            query, intent, confidence, margin
        )
        if not passed:
            return {
                "answer": clarification,
                "sources": [],
                "debug": {"stage": "guardrail_pre", "reason": reason, "intent": intent}
            }

        index_names = ROUTING_MAP.get(intent, [])
        if not index_names:
            return {
                "answer": "I'm not sure what you're asking. Can you clarify?",
                "sources": [],
                "debug": {"stage": "routing", "intent": intent}
            }

        qvec = self._embed_query(query)
        results = {}
        if "project" in index_names:
            results["project"] = self.project_idx.search(qvec, top_k=15)
        if "general" in index_names:
            results["general"] = self.general_idx.search(qvec, top_k=8)

        chunks = self._fuse_rrf(results) if len(results) > 1 else list(results.values())[0]

        relevant, fallback = self.guardrails.post_rag_check(chunks)
        if not relevant:
            return {"answer": fallback, "sources": [], "debug": {"stage": "guardrail_post"}}

        result = self.generator.generate(query, chunks)
        result["debug"] = {
            "intent": intent,
            "confidence": confidence,
            "margin": margin,
            "indexes": index_names,
        }
        return result


# ═════════════════════════════════════════════════════
# 7. GRADIO UI
# ═════════════════════════════════════════════════════
print("🌿 Starting EcoGuide AI Local Gradio App...")
rag = EcoGuideRAG()


def respond(message, history):
    if not message or not message.strip():
        return "Please enter a question."

    try:
        result = rag.answer(message.strip())
        answer = result["answer"]

        if result.get("sources"):
            srcs = "\n".join(f"• {s}" for s in result["sources"] if s.strip())
            if srcs.strip():
                answer += f"\n\n📚 **Sources:**\n{srcs}"

        debug = result.get("debug", {})
        if debug:
            answer += f"\n\n<sub>🎯 {debug.get('intent')} | confidence: {debug.get('confidence', 0):.2f} | margin: {debug.get('margin', 0):.2f}</sub>"

        return answer
    except Exception as e:
        return f"❌ Error: {str(e)}"



🌿 Starting EcoGuide AI Local Gradio App...
[RAG] Initializing...
[Classifier] Loading from best intent classifier ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[Classifier] Ready on cuda | 8 labels


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[Retriever:project] Loading from ecoguide\data\project_index.faiss ...
[Retriever:project] Ready: 180 vectors
[Retriever:general] Loading from ecoguide\data\general_index.faiss ...
[Retriever:general] Ready: 102 vectors
[RAG] Ready.


    def answer(self, query: str) -> dict:
        # ── Keyword override for project name ──────────────────────
        lower = query.lower()
        if "premium sortify" in lower or "بريميم" in lower or "سورتيفاي" in lower or "sortify" in lower:
            classification = {
                "intent": "project_information",
                "confidence": 0.999,
                "margin": 0.999,
                "all_scores": {}
            }
            intent = "project_information"
            confidence = 0.999
            margin = 0.999
            print(f"[RAG] Keyword override: project_information (Premium Sortify detected)")
        else:
            classification = self.classifier.classify(query)
            intent = classification["intent"]
            confidence = classification["confidence"]
            margin = classification["margin"]
        # ── End override ───────────────────────────────────────────
        
        print(f"[RAG] {intent} | conf={confidence:.3f} | margin={margin:.3f}")
        # ... rest of the method stays exactly the same

In [3]:
print("🌿 EcoGuide AI is running at http://localhost:7860"  )

🌿 EcoGuide AI is running at http://localhost:7860


In [4]:
import socket

demo = gr.ChatInterface(
    respond,
    title="🌿 EcoGuide AI — Premium Sortify (Local)",
    description=(
        "Multilingual Sustainability Assistant for Cairo University. "
        "Ask about Premium Sortify, recycling, or sustainability in **English** or **العربية**.\n\n"
        "Running locally on your GPU — instant responses, no internet required."
    ),
    examples=[
        "What is Premium Sortify?",
        "كيف تعمل الحاوية الذكية؟",
        "Who are the developers?",
        "Why is recycling plastic important?",
        "How do I recycle plastic on campus and why does it matter?",
        "ما هو التدوير؟",
    ],
    cache_examples=False,
)

if __name__ == "__main__":
    def get_free_port() -> int:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.bind(("127.0.0.1", 0))
            return s.getsockname()[1]

    demo.launch(
        share=False,
        server_name="0.0.0.0",
        server_port=get_free_port(),
    )

* Running on local URL:  http://0.0.0.0:61363
* To create a public link, set `share=True` in `launch()`.


In [5]:
clf = EcoGuideIntentClassifier()

[Classifier] Loading from best intent classifier ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[Classifier] Ready on cuda | 8 labels


In [6]:
clf.classify("tell me everything you know about premium sortify")

{'intent': 'project_information',
 'confidence': 0.9603122472763062,
 'margin': 0.9236315488815308,
 'all_scores': {'out_of_scope': 0.036680709570646286,
  'project_information': 0.9603122472763062,
  'recycle_glass': 0.0008278111927211285,
  'recycle_metal': 0.00047477803309448063,
  'recycle_organic': 0.0003318357630632818,
  'recycle_plastic': 0.00040906123467721045,
  'sustainability_definition': 0.0006269353325478733,
  'waste_sorting': 0.00033665678347460926}}

In [7]:
clf.classify("what is your information about premium Sortify?")

{'intent': 'out_of_scope',
 'confidence': 0.7656996250152588,
 'margin': 0.5373739004135132,
 'all_scores': {'out_of_scope': 0.7656996250152588,
  'project_information': 0.2283257395029068,
  'recycle_glass': 0.0018509479705244303,
  'recycle_metal': 0.0009578907047398388,
  'recycle_organic': 0.0006631226860918105,
  'recycle_plastic': 0.0007371990941464901,
  'sustainability_definition': 0.0008848810684867203,
  'waste_sorting': 0.0008806157857179642}}

In [8]:
clf.classify("sustanable development")

{'intent': 'sustainability_definition',
 'confidence': 0.9978663325309753,
 'margin': 0.9974163174629211,
 'all_scores': {'out_of_scope': 0.0002708813117351383,
  'project_information': 0.0004499854985624552,
  'recycle_glass': 0.0001752876996761188,
  'recycle_metal': 0.0003529045497998595,
  'recycle_organic': 0.00036060260026715696,
  'recycle_plastic': 0.0003253388567827642,
  'sustainability_definition': 0.9978663325309753,
  'waste_sorting': 0.00019862566841766238}}

In [9]:
clf.classify("ecogide")


{'intent': 'project_information',
 'confidence': 0.9979090690612793,
 'margin': 0.9974202513694763,
 'all_scores': {'out_of_scope': 0.00027793916524387896,
  'project_information': 0.9979090690612793,
  'recycle_glass': 0.0004888215917162597,
  'recycle_metal': 0.0001635337102925405,
  'recycle_organic': 0.0002669934183359146,
  'recycle_plastic': 0.00028841133462265134,
  'sustainability_definition': 0.0003747288428712636,
  'waste_sorting': 0.00023058211081661284}}

In [10]:
clf.classify("what is premum sortify")

{'intent': 'project_information',
 'confidence': 0.9980352520942688,
 'margin': 0.9975403547286987,
 'all_scores': {'out_of_scope': 0.0002428821026114747,
  'project_information': 0.9980352520942688,
  'recycle_glass': 0.00039595644921064377,
  'recycle_metal': 0.0002163359458791092,
  'recycle_organic': 0.00020069263700861484,
  'recycle_plastic': 0.00020608262275345623,
  'sustainability_definition': 0.0004948988207615912,
  'waste_sorting': 0.0002079822006635368}}

In [11]:
clf.classify("how does eco guide work")

{'intent': 'project_information',
 'confidence': 0.9978189468383789,
 'margin': 0.9972741007804871,
 'all_scores': {'out_of_scope': 0.00016576248162891716,
  'project_information': 0.9978189468383789,
  'recycle_glass': 0.00040120608173310757,
  'recycle_metal': 0.00016779643192421645,
  'recycle_organic': 0.0002672176342457533,
  'recycle_plastic': 0.00022426378563977778,
  'sustainability_definition': 0.0005448553129099309,
  'waste_sorting': 0.00040999031625688076}}

In [12]:
clf.classify("what is sustanability")

{'intent': 'sustainability_definition',
 'confidence': 0.9979972243309021,
 'margin': 0.9976509213447571,
 'all_scores': {'out_of_scope': 0.000285484769847244,
  'project_information': 0.0002879827516153455,
  'recycle_glass': 0.00023047628928907216,
  'recycle_metal': 0.0003347601159475744,
  'recycle_organic': 0.00033851322950795293,
  'recycle_plastic': 0.00034627970308065414,
  'sustainability_definition': 0.9979972243309021,
  'waste_sorting': 0.00017929494788404554}}

In [13]:
respond("1. How does Premium Sortify recycle plastic?","none")

[RAG] Keyword override: project_information (Premium Sortify detected)
[RAG] project_information | conf=0.999 | margin=0.999
[RAG] recycle_plastic | conf=0.996 | margin=0.995
[Generator] Loading Qwen/Qwen2.5-7B-Instruct ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[Generator] Ready.


'Premium Sortify recycles plastic by intercepting it at the point of disposal and routing it into a closed-loop recycling pipeline. This process helps in reducing the plastic waste sent to sanitary landfills by more than 70%. The sorted plastics, such as HDPE and PP, are then processed into filament or interlock tiles, which further reduces greenhouse gas emissions by 71% compared to producing virgin plastic.\n\n📚 **Sources:**\n• Competition_Form | Environmental Impact / GHG Reduction — HDPE and PP\n• EcoGuide_Awareness | Awareness / EcoGuide Mission\n• Competition_Form | Environmental Impact / Visual and Health Benefits\n• EcoGuide_AI_Project_Documentation.md + Competition_Form | Environmental Impact / Landfill Reduction\n• EcoGuide_Awareness | Awareness / Plastic Pollution Impact\n\n<sub>🎯 recycle_plastic | confidence: 1.00 | margin: 0.99</sub>'

In [14]:

def respond(message, history):
    if not message or not message.strip():
        return "Please enter a question."
    print(f"Q {message}")
    try:
        result = rag.answer(message.strip())
        answer = result["answer"]

        if result.get("sources"):
            srcs = "\n".join(f"• {s}" for s in result["sources"] if s.strip())
            if srcs.strip():
                answer += f"\n\n📚 **Sources:**\n{srcs}"

        debug = result.get("debug", {})
        if debug:
            answer += f"\n\n<sub>🎯 {debug.get('intent')} | confidence: {debug.get('confidence', 0):.2f} | margin: {debug.get('margin', 0):.2f}</sub>"
        print(answer)
        return answer
    except Exception as e:
        return f"❌ Error: {str(e)}"



In [15]:
TEST_CASES = ["1. How does Premium Sortify recycle plastic?", "2. Why is the project sustainable?", "3. What glass does the smart bin accept?"]
for message in TEST_CASES:
    respond(message, "d")

Q 1. How does Premium Sortify recycle plastic?
[RAG] Keyword override: project_information (Premium Sortify detected)
[RAG] project_information | conf=0.999 | margin=0.999
[RAG] recycle_plastic | conf=0.996 | margin=0.995
Premium Sortify recycles plastic by intercepting it at the point of disposal and routing it into a closed-loop recycling pipeline. This process helps in reducing the plastic waste sent to sanitary landfills by more than 70%. The sorted plastic is then processed into filament or interlock tiles, which cuts greenhouse gas emissions by 71% compared to producing virgin plastic.

📚 **Sources:**
• Competition_Form | Environmental Impact / GHG Reduction — HDPE and PP
• EcoGuide_Awareness | Awareness / EcoGuide Mission
• Competition_Form | Environmental Impact / Visual and Health Benefits
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Environmental Impact / Landfill Reduction
• EcoGuide_Awareness | Awareness / Plastic Pollution Impact

<sub>🎯 recycle_plastic | co

In [16]:
TEST_CASES=["4. How is metal sorted in the system?", "5. Who designed the sorting algorithm?", "6. كيف تعمل الحاوية الذكية؟", "7. من هم المطورون؟",
              ]
for message in TEST_CASES:
    respond(message, "d")


Q 4. How is metal sorted in the system?
[RAG] recycle_metal | conf=0.997 | margin=0.996
[RAG] recycle_metal | conf=0.997 | margin=0.996
In the system, metal is sorted using an AI waste classification model (ResNet50) which classifies the material into metal along with other categories such as glass, organic waste, and plastic. Once classified, a rotating disk routes the metal item to the correct internal chamber for further processing.

📚 **Sources:**
• EPA_Waste_Hierarchy | Waste Hierarchy / Overview
• EcoGuide_AI_Project_Documentation.md | Overall System Workflow
• EcoGuide_AI_Project_Documentation.md | AI Waste Classifier / Waste Taxonomy
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Robotic Arm / Quality Assurance for Filament Production
• EcoGuide_Campus_Sustainability | Cairo University / E-Waste Collection

<sub>🎯 recycle_metal | confidence: 1.00 | margin: 1.00</sub>
Q 5. Who designed the sorting algorithm?
[RAG] project_information | conf=0.893 | margin=0.791
[RAG

In [17]:
TEST_CASES=["8. لماذا يعتبر التدوير مهمًا؟", "9. ما هي فكرة المشروع؟", "10. كيف أفرز المخلفات في الحرم؟", "11. What is الاستدامة؟", "12. Who are الـ developers في الفريق؟"
              ]
for message in TEST_CASES:
    respond(message, "d")

Q 8. لماذا يعتبر التدوير مهمًا؟
[RAG] sustainability_definition | conf=0.998 | margin=0.997
[RAG] sustainability_definition | conf=0.998 | margin=0.997
التدوير مهم لأسباب عديدة، منها:

1. توفير الطاقة: تدوير بعض المواد مثل الألمنيوم يوفر كميات هائلة من الطاقة مقارنة بإنتاج المواد الخام الجديدة. على سبيل المثال، تدوير علبة ألمنيوم واحدة يمكن أن توفر طاقة كافية لتشغيل حاسوب محمول لمدة 11 ساعة.

2. الحفاظ على الموارد الطبيعية: تدوير الورق يمكن أن يحمي شجرة ناضجة كل طن من الورق يتم تدويره، كما أنه يوفر كميات كبيرة من المياه والطاقة.

3. تقليل التلوث: تقليل استخدام المواد الخام الطازجة وتحويل النفايات إلى منتجات جديدة يقلل من الحاجة إلى التعدين والتصنيع، مما يساهم في تقليل الانبعاثات والتلوث.

4. الحفاظ على التنوع البيولوجي: تدوير الورق ليس فقط يحمي الغابات، بل يساهم أيضاً في الحفاظ على التنوع البيولوجي من خلال الحفاظ على الموائل الطبيعية للأنواع المختلفة.

5. توفير الطاقة في تدوير المعادن: تدوير المعادن مثل الألمنيوم والصلب يوفر كميات هائلة من الطاقة مقارنة بإنتاجها من خامات الطبيعية. على 

In [18]:
TEST_CASES=["13. How does الحاوية الذكية بتعمل؟", "14. Tell me عن فكرة بريميم سورتيفاي", "15. What are أهداف التنمية المستدامة؟", 
            "16. premum sortify", "what is 17. ecogide",]

for message in TEST_CASES:
    respond(message, "d")

Q 13. How does الحاوية الذكية بتعمل؟
[RAG] project_information | conf=0.997 | margin=0.995
[RAG] project_information | conf=0.997 | margin=0.995
الحاوية الذكية تعمل وفقًا لسير عمل شامل وآلي:

1. يضع المستخدم المخلف عبر الفتحة العلوية.
2. يكتشف حساس الأشعة تحت الحمراء العنصر.
3. تلتقط الكاميرا صورة له.
4. يتم تصنيف المادة بواسطة نموذج الذكاء الاصطناعي ResNet50 في غضون مللي ثانية.
5. يأمر ESP32 محرك التدرج بتدوير القرص الدوار.
6. يسقط العنصر في الحجرة المخصصة له.
7. تسجل بيانات مستوى الامتلاء على لوحة التحكم السحابية.

هذا السير العمل الآلي يساعد في تصنيف وتوجيه المخلفات بشكل دقيق وفعال.

📚 **Sources:**
• EcoGuide_AI_Project_Documentation.md + Competition_Form | إيكو جايد AI / الأسئلة المدعومة
• مسابقة_إدارة_المخلفات_الجامعية.md | الذراع الآلية / التكامل مع خط الحاوية الذكية
• EcoGuide_AI_Project_Documentation.md | IoT and Monitoring Workflow
• مسابقة_إدارة_المخلفات_الجامعية.md | الحاوية الذكية / الإلكترونيات التحكم
• مسابقة_إدارة_المخلفات_الجامعية.md | النموذج الأولي / مسار التقنية الذك

Q 13. How does الحاوية الذكية بتعمل؟
[RAG] project_information | conf=0.997 | margin=0.995
[RAG] project_information | conf=0.997 | margin=0.995
الحاوية الذكية تعمل وفقًا لسير عمل شامل وآلي:

1. يضع المستخدم المخلف عبر الفتحة العلوية.
2. يكتشف حساس الأشعة تحت الحمراء وجود المخلف.
3. تلتقط الكاميرا صورة للمخلف.
4. يتم تصنيف المادة بواسطة نموذج الذكاء الاصطناعي ResNet50 في غضون مللي ثانية.
5. يدير ESP32 محرك التدرج لتدوير القرص الدوار.
6. يسقط العنصر في الحجرة المخصصة له.
7. تسجل بيانات مستوى الامتلاء على لوحة التحكم السحابية.

هذه العملية تسمح بتصنيف وتوجيه المخلفات بشكل آلي وفعال، مما يساهم في إدارة المخلفات بشكل أكثر استدامة.

📚 **Sources:**
• EcoGuide_AI_Project_Documentation.md | IoT and Monitoring Workflow
• مسابقة_إدارة_المخلفات_الجامعية.md | الحاوية الذكية / وحدة الكاميرا
• مسابقة_إدارة_المخلفات_الجامعية.md | الحاوية الذكية / الحجرات الداخلية
• مسابقة_إدارة_المخلفات_الجامعية.md | الحاوية الذكية / البنية المعمارية
• EcoGuide_AI_Project_Documentation.md + Competition_Form | إيكو جايد AI / الأسئلة المدعومة
• مسابقة_إدارة_المخلفات_الجامعية.md | تطبيق الهاتف المحمول / التكامل مع إيكو جايد AI
• مسابقة_إدارة_المخلفات_الجامعية.md | الحاوية الذكية / سير العمل الشامل
• مسابقة_إدارة_المخلفات_الجامعية.md | الحاوية الذكية / بنية النشر في الحرم الجامعي
• مسابقة_إدارة_المخلفات_الجامعية.md | تطبيق الهاتف المحمول / نظام النقاط
• مسابقة_إدارة_المخلفات_الجامعية.md | الذراع الآلية / النموذج الثانوي للذكاء الاصطناعي
• مسابقة_إدارة_المخلفات_الجامعية.md | المساعد الرقمي الذكي
• مسابقة_إدارة_المخلفات_الجامعية.md | الذراع الآلية / التكامل مع خط الحاوية الذكية
• مسابقة_إدارة_المخلفات_الجامعية.md | الذراع الآلية / معالجة العناصر المتعددة
• مسابقة_إدارة_المخلفات_الجامعية.md | الحاوية الذكية / الإلكترونيات التحكم
• مسابقة_إدارة_المخلفات_الجامعية.md | النموذج الأولي / مسار التقنية الذكية

<sub>🎯 project_information | confidence: 1.00 | margin: 1.00</sub>
Q 14. Tell me عن فكرة بريميم سورتيفاي
[RAG] Keyword override: project_information (Premium Sortify detected)
[RAG] project_information | conf=0.999 | margin=0.999
[RAG] project_information | conf=0.998 | margin=0.998
فكرة بريميم سورتيفاي هي تقديم منظومة متكاملة لإدارة المخلفات البلاستيكية داخل الجامعة. يتم تحقيق ذلك من خلال استخدام تقنيات الذكاء الاصطناعي، وإنترنت الأشياء، والروبوتات في حاويات فرز ذكية. المشروع يستهدف النوع الأكثر انتشارًا من المخلفات البلاستيكية بنسبة 55% وفقًا لاستبيان ميداني. يتم توجيه البلاستيك المفرز إلى خط إعادة تدوير مغلق، مما ينتج عنه خيوط الطباعة ثلاثية الأبعاد وبلاط الإنترلوك. هذا النهج يوفر مرونة اقتصادية حيث يمكن توجيه الإنتاج نحو المنتج الأكثر ربحية أو طلباً في السوق في كل دورة تشغيلية. بالإضافة إلى ذلك، يهدف المشروع إلى تعزيز ثقافة الاستدامة والتغيير السلوكي لدى الطلاب، مما يجعل كل قرار تخلص مرئياً ومكافأً وتعليمياً.

📚 **Sources:**
• مسابقة_إدارة_المخلفات_الجامعية.md | عملية إعادة التدوير / نموذج الاقتصاد الدائري
• EcoGuide_AI_Project_Documentation.md | إيكو جايد AI / نظام RAG والاسترجاع
• مسابقة_إدارة_المخلفات_الجامعية.md | بيانات الفريق الطلابي
• مسابقة_إدارة_المخلفات_الجامعية.md | الحاوية الذكية / بنية النشر في الحرم الجامعي
• مسابقة_إدارة_المخلفات_الجامعية.md | بيانات الفريق / التعاون متعدد التخصصات
• مسابقة_إدارة_المخلفات_الجامعية.md | الأثر البيئي / الفوائد البصرية والصحية
• EcoGuide_AI_Project_Documentation.md + Competition_Form | إيكو جايد AI / نظرة عامة
• مسابقة_إدارة_المخلفات_الجامعية.md | بيانات الفريق / مطور تطبيق الهاتف المحمول
• مسابقة_إدارة_المخلفات_الجامعية.md | الأثر البيئي / ثقافة الاستدامة والتغيير السلوكي
• مسابقة_إدارة_المخلفات_الجامعية.md | الأثر البيئي / خفض المدافن
• مسابقة_إدارة_المخلفات_الجامعية.md | وصف فكرة المشروع
• مسابقة_إدارة_المخلفات_الجامعية.md | بيانات الفريق / قائد تطوير الذكاء الاصطناعي
• مسابقة_إدارة_المخلفات_الجامعية.md | بيانات قائد الفريق
• مسابقة_إدارة_المخلفات_الجامعية.md | بيانات الفريق / نظرة عامة
• مسابقة_إدارة_المخلفات_الجامعية.md | بيانات الفريق / المسابقة

<sub>🎯 project_information | confidence: 1.00 | margin: 1.00</sub>
Q 15. What are أهداف التنمية المستدامة؟
[RAG] sustainability_definition | conf=0.998 | margin=0.998
[RAG] sustainability_definition | conf=0.998 | margin=0.998
أهداف التنمية المستدامة تهدف إلى تلبية احتياجاتنا الحالية دون التأثير على قدرة الأجيال القادمة على تلبية احتياجاتها. هذه الأهداف تشمل ثلاثة ركائز رئيسية وهي حماية البيئة، تحقيق العدالة الاجتماعية، وتحقيق الجدوى الاقتصادية. تعمل هذه الأهداف على تعزيز الاستدامة من خلال تقليل المخلفات، الحفاظ على الموارد الطبيعية، وتقليل التلوث.

📚 **Sources:**
• EPA_Waste_Hierarchy | هرم المخلفات / نظرة عامة
• General_Sustainability_Reference | الاستدامة / التعريف
• General_Recycling_Reference | إعادة التدوير / الأساسيات
• EcoGuide_Awareness | التوعية / العدالة بين الأجيال
• General_Sustainability_Reference | Sustainability / Definition
• EcoGuide_Awareness | Awareness / Green Careers
• EcoGuide_Awareness | التوعية / الوظائف الخضراء
• EcoGuide_Awareness | Awareness / Intergenerational Justice

<sub>🎯 sustainability_definition | confidence: 1.00 | margin: 1.00</sub>
Q 16. premum sortify
[RAG] Keyword override: project_information (Premium Sortify detected)
[RAG] project_information | conf=0.999 | margin=0.999
[RAG] project_information | conf=0.998 | margin=0.998
Premium Sortify is an integrated campus waste management system developed by a team of five students from Cairo University Faculty of Engineering. It focuses on plastic waste, which makes up 55% of the surveyed campus waste, by using smart sorting bins that combine artificial intelligence, Internet of Things, and robotics. These bins operate autonomously and are placed across campus buildings, cafeterias, and common areas. The system aims to reduce plastic waste sent to landfills by over 70%, cultivate sustainable habits among students, and promote a circular economy through the production of 3D-printing filament and interlock tiles from sorted plastics.

📚 **Sources:**
• EcoGuide_AI_Project_Documentation.md | Smart Bin / Campus Deployment Architecture
• Competition_Form | Recycling Process / Circular Economy Model
• Competition_Form | Team Information
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Mobile Application / Overview
• Competition_Form | Team Information / Overview
• Competition_Form | Team Information / Interdisciplinary Collaboration
• Competition_Form | Environmental Impact / Sustainability Culture and Behavioral Change
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Smart Bin / Architecture
• Competition_Form | Team Information / Abdelrahman Sayed
• EcoGuide_AI_Project_Documentation.md | EcoGuide AI Overview
• Competition_Form | Competition Context
• Competition_Form | Environmental Impact / Visual and Health Benefits
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Environmental Impact / Landfill Reduction
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Project Overview
• Competition_Form | Team Information / AI Development Lead

<sub>🎯 project_information | confidence: 1.00 | margin: 1.00</sub>
Q what is 17. ecogide
[RAG] project_information | conf=0.998 | margin=0.998
[RAG] project_information | conf=0.998 | margin=0.998
It seems there might be a typo in your question. Based on the context provided, EcoGuide AI is the bilingual conversational assistant layer of Premium Sortify. If you meant to ask about a specific aspect related to EcoGuide AI, could you please clarify your question? For example, you might want to know more about its functionality, how it operates, or its role in Premium Sortify.

📚 **Sources:**
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Mobile Application / EcoGuide AI Integration
• EcoGuide_AI_Project_Documentation.md | EcoGuide AI / Distinction from AI Waste Classifier
• EcoGuide_AI_Project_Documentation.md + Competition_Form | EcoGuide AI / Supported Questions
• EcoGuide_AI_Project_Documentation.md + Competition_Form | EcoGuide AI / Overview
• Competition_Form | Team Information / Abdelrahman Sayed
• EcoGuide_AI_Project_Documentation.md | EcoGuide AI / Live Data and Tool-Calling Integration
• EcoGuide_AI_Project_Documentation.md | EcoGuide AI / Intent Classification Architecture
• EcoGuide_AI_Project_Documentation.md | EcoGuide AI / Guardrail System
• EcoGuide_AI_Project_Documentation.md | EcoGuide AI Overview
• EcoGuide_AI_Project_Documentation.md | EcoGuide AI / RAG and Retrieval System
• EcoGuide_AI_Project_Documentation.md + Competition_Form | EcoGuide AI / Sustainability Education
• Competition_Form | EcoGuide AI / Value Proposition and Availability
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Environmental Impact / Circular Economy and Resource Conservation
• Competition_Form | Team Information / AI Development Lead
• Competition_Form | Environmental Impact / GHG Reduction — HDPE and PP

<sub>🎯 project_information | confidence: 1.00 | margin: 1.00</sub>

In [19]:
respond("15. What are أهداف التنمية المستدامة؟","v")

Q 15. What are أهداف التنمية المستدامة؟
[RAG] sustainability_definition | conf=0.998 | margin=0.998
[RAG] sustainability_definition | conf=0.998 | margin=0.998
أهداف التنمية المستدامة تهدف إلى تلبية احتياجاتنا الحالية دون التأثير على قدرة الأجيال القادمة على تلبية احتياجاتها. هذه الأهداف تشمل ثلاثة ركائز رئيسية وهي حماية البيئة، والعدالة الاجتماعية، والجدوى الاقتصادية. من خلال تحقيق هذه الأهداف، نسعى إلى الحفاظ على الكوكب وتحسين حياة البشر بطريقة مستدامة.

📚 **Sources:**
• EcoGuide_Awareness | التوعية / العدالة بين الأجيال
• EcoGuide_Awareness | Awareness / Green Careers
• EcoGuide_Awareness | التوعية / الوظائف الخضراء
• EPA_Waste_Hierarchy | هرم المخلفات / نظرة عامة
• EcoGuide_Awareness | Awareness / Intergenerational Justice
• General_Sustainability_Reference | Sustainability / Definition
• General_Sustainability_Reference | الاستدامة / التعريف
• General_Recycling_Reference | إعادة التدوير / الأساسيات

<sub>🎯 sustainability_definition | confidence: 1.00 | margin: 1.00</sub>


'أهداف التنمية المستدامة تهدف إلى تلبية احتياجاتنا الحالية دون التأثير على قدرة الأجيال القادمة على تلبية احتياجاتها. هذه الأهداف تشمل ثلاثة ركائز رئيسية وهي حماية البيئة، والعدالة الاجتماعية، والجدوى الاقتصادية. من خلال تحقيق هذه الأهداف، نسعى إلى الحفاظ على الكوكب وتحسين حياة البشر بطريقة مستدامة.\n\n📚 **Sources:**\n• EcoGuide_Awareness | التوعية / العدالة بين الأجيال\n• EcoGuide_Awareness | Awareness / Green Careers\n• EcoGuide_Awareness | التوعية / الوظائف الخضراء\n• EPA_Waste_Hierarchy | هرم المخلفات / نظرة عامة\n• EcoGuide_Awareness | Awareness / Intergenerational Justice\n• General_Sustainability_Reference | Sustainability / Definition\n• General_Sustainability_Reference | الاستدامة / التعريف\n• General_Recycling_Reference | إعادة التدوير / الأساسيات\n\n<sub>🎯 sustainability_definition | confidence: 1.00 | margin: 1.00</sub>'

In [20]:
TEST_CASES=["explain sustanable campus", "21. What doesn't the smart bin accept?", "22. Which materials can't be recycled?"]


for message in TEST_CASES:
    respond(message, "d")

Q explain sustanable campus
[RAG] sustainability_definition | conf=0.998 | margin=0.997
[RAG] sustainability_definition | conf=0.998 | margin=0.997
A sustainable campus is an educational institution that integrates environmental, economic, and social responsibility into its operations and culture. At Cairo University, sustainability initiatives like tree canopy expansion, bicycle infrastructure, and composting programs aim to reduce the campus's ecological footprint while enhancing the well-being of students and staff.

For example, expanding the tree canopy with native species helps cool the environment, improve air quality, and create wildlife habitats. Implementing bicycle lanes and a bike-sharing program reduces reliance on cars, lowering air pollution and greenhouse gas emissions. Establishing a centralized composting program turns organic waste into valuable compost, reducing landfill waste and providing free fertilizer for campus gardens.

These efforts not only contribute to a 

Q explain sustanable campus
[RAG] sustainability_definition | conf=0.998 | margin=0.997
[RAG] sustainability_definition | conf=0.998 | margin=0.997
A sustainable campus is an educational institution that integrates environmental stewardship, economic viability, and social equity into its operations and culture. At Cairo University, sustainability initiatives such as expanding tree canopies, promoting bicycle infrastructure, and implementing composting programs contribute to a more sustainable campus. These efforts aim to reduce the institution's ecological footprint, enhance the well-being of students and staff, and foster a culture of environmental responsibility. For example, tree canopy expansion provides natural cooling and absorbs pollutants, while bicycle lanes and a bike-sharing program reduce carbon emissions and promote healthier transportation options. Centralized composting diverts organic waste from landfills, turning it into valuable resources for landscaping. Additionally, sustainable cafeteria practices, such as sourcing locally and offering plant-based meals, reduce the carbon footprint associated with food production and consumption. Car-free days encourage walking and cycling, improving air quality and public health. Through these and other initiatives, Cairo University strives to create a model of sustainable living and learning that benefits both the campus community and the broader environment.

📚 **Sources:**
• EcoGuide_Campus_Sustainability | Cairo University / Sustainable Cafeteria
• EcoGuide_Awareness | Awareness / Urban Heat Islands
• EcoGuide_Campus_Sustainability | Cairo University / Car-Free Campus Days
• EcoGuide_Awareness | Awareness / Food Waste & Climate
• EcoGuide_Campus_Sustainability | Cairo University / Campus Composting
• EcoGuide_Campus_Sustainability | Cairo University / Tree Canopy Expansion
• EcoGuide_Awareness | Awareness / Water Refill Stations
• EcoGuide_Campus_Sustainability | Cairo University / Bicycle Infrastructure

<sub>🎯 sustainability_definition | confidence: 1.00 | margin: 1.00</sub>
Q 21. What doesn't the smart bin accept?
[RAG] waste_sorting | conf=0.998 | margin=0.997
[RAG] waste_sorting | conf=0.998 | margin=0.997
The context provided does not explicitly state what the smart bin does not accept. However, based on the information given, the smart bin primarily accepts waste items that can be classified into four main categories: Glass, Metal, Organic Waste, and Plastic. Items that fall outside these categories or those that cause the ResNet50 classifier to return a confidence score below the routing threshold would be considered uncertain and thus not directly accepted by the smart bin without further processing. Therefore, it can be inferred that the smart bin does not accept items that cannot be reliably classified into one of the four main categories or those that are too complex or mixed to be handled by the initial classification system.

📚 **Sources:**
• EcoGuide_AI_Project_Documentation.md | Overall System Workflow
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Mobile Application / Points System
• EcoGuide_AI_Project_Documentation.md | Robotic Arm / Secondary AI Model
• Competition_Form | Smart Bin / Control Electronics
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Smart Bin / Camera Module
• EcoGuide_AI_Project_Documentation.md | AI Waste Classifier / Waste Taxonomy
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Smart Bin / Architecture
• EcoGuide_AI_Project_Documentation.md + Competition_Form | EcoGuide AI / Supported Questions
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Robotic Arm / Multi-Item Handling
• Competition_Form | Environmental Impact / Visual and Health Benefits
• EcoGuide_AI_Project_Documentation.md | Robotic Arm / Integration with Smart Bin Pipeline
• Competition_Form | Team Information / Hardware Development Team
• EcoGuide_AI_Project_Documentation.md | Robotic Arm / Uncertainty Handling
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Robotic Arm / Central Sorting Station
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Mobile Application / EcoGuide AI Integration

<sub>🎯 waste_sorting | confidence: 1.00 | margin: 1.00</sub>
Q 22. Which materials can't be recycled?
[RAG] waste_sorting | conf=0.995 | margin=0.994
[RAG] waste_sorting | conf=0.995 | margin=0.994
Based on the provided context, the materials that cannot be recycled within the described system are not explicitly mentioned. However, the system focuses on recycling Glass, Metal, Organic Waste, and Plastic. Therefore, it can be inferred that other materials not categorized under these four streams (such as electronic waste, textiles, or certain hazardous materials) would likely not be recycled through this particular pipeline.

📚 **Sources:**
• EcoGuide_AI_Project_Documentation.md | Smart Bin / Internal Compartments
• Competition_Form | Recycling Process / Thermal Reshaping Properties
• EcoGuide_AI_Project_Documentation.md | EcoGuide AI / Intent Classification Architecture
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Recycling Process / Complete Pipeline
• Competition_Form | Environmental Impact / Campus Waste Baseline
• Competition_Form | Environmental Impact / GHG Reduction — HDPE and PP
• Competition_Form | Mobile Application / Achievements and Badges
• Competition_Form | Recycling Process / PET
• Competition_Form | Environmental Impact / Energy Savings — HDPE and PP
• EcoGuide_AI_Project_Documentation.md | AI Waste Classifier / Waste Taxonomy
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Robotic Arm / Quality Assurance for Filament Production
• Competition_Form | Environmental Impact / CO₂ and Greenhouse Gas Reduction
• EcoGuide_AI_Project_Documentation.md | Recycling Pipeline and Closed-Loop System
• EcoGuide_AI_Project_Documentation.md + Competition_Form | Robotic Arm / Plastic Subtype Sorting by Resin
• EcoGuide_AI_Project_Documentation.md | Project Objectives

<sub>🎯 waste_sorting | confidence: 1.00 | margin: 0.99</sub>

In [ ]:
TEST_CASES=["23. Why isn't plastic biodegradable?",
              "24. What is the capital of France?", "25. How do I cook pasta?"]

for message in TEST_CASES:
    respond(message, "d")

Q 23. Why isn't plastic biodegradable?
[RAG] sustainability_definition | conf=0.998 | margin=0.997
[RAG] sustainability_definition | conf=0.998 | margin=0.997
Plastic isn't biodegradable because it is made from synthetic polymers that do not break down easily through natural processes. Unlike organic materials which decompose with the help of bacteria and other microorganisms, plastic takes a very long time to degrade due to its chemical structure. This leads to persistent pollution in the environment, including in oceans and landfills.

📚 **Sources:**
• EcoGuide_Awareness | Awareness / Water Refill Stations
• EcoGuide_Awareness | Awareness / Microplastics & Health
• EcoGuide_Awareness | Awareness / Zero Waste Lifestyle
• EcoGuide_Awareness | Awareness / EcoGuide Mission
• General_Plastic_Reference | Plastic Types / PP
• General_Environmental_Reference | Landfills / Environmental Impact
• General_Plastic_Reference | Plastic Types / PET
• EcoGuide_Awareness | Awareness / Plastic Polluti

[RAG] Keyword override: project_information (Premium Sortify detected)
[RAG] project_information | conf=0.999 | margin=0.999
[RAG] project_information | conf=0.998 | margin=0.998
[RAG] Keyword override: project_information (Premium Sortify detected)
[RAG] project_information | conf=0.999 | margin=0.999
[RAG] project_information | conf=0.998 | margin=0.997
[RAG] sustainability_definition | conf=0.998 | margin=0.998
[RAG] sustainability_definition | conf=0.998 | margin=0.998


Q 23. Why isn't plastic biodegradable?
[RAG] sustainability_definition | conf=0.998 | margin=0.997
[RAG] sustainability_definition | conf=0.998 | margin=0.997
Plastic is not biodegradable because it is made from synthetic polymers that do not break down easily through natural processes. Unlike organic materials which decompose through biological processes involving microorganisms, plastic's structure is too complex and stable for most bacteria and fungi to break down efficiently. This stability is what makes plastic useful for its intended purposes, such as durability and protection of products, but it also leads to long-lasting environmental impacts when not managed properly, contributing to issues like ocean pollution and landfill overload.

📚 **Sources:**
• General_Plastic_Reference | Plastic Types / PET
• General_Plastic_Reference | Plastic Types / PP
• EcoGuide_Awareness | Awareness / Microplastics & Health
• EcoGuide_Awareness | Awareness / EcoGuide Mission
• EcoGuide_Awareness | Awareness / Water Refill Stations
• General_Environmental_Reference | Landfills / Environmental Impact
• EcoGuide_Awareness | Awareness / Plastic Pollution Impact
• EcoGuide_Awareness | Awareness / Zero Waste Lifestyle

<sub>🎯 sustainability_definition | confidence: 1.00 | margin: 1.00</sub>
Q 24. What is the capital of France?
[RAG] out_of_scope | conf=0.998 | margin=0.997
[RAG] out_of_scope | conf=0.998 | margin=0.997
I'm not sure I understood your question. Could you rephrase or specify whether you're asking about Premium Sortify's features or general recycling information?

<sub>🎯 out_of_scope | confidence: 0.00 | margin: 0.00</sub>
Q 25. How do I cook pasta?
[RAG] recycle_organic | conf=0.996 | margin=0.994
[RAG] recycle_organic | conf=0.996 | margin=0.994
I'm sorry, but the provided context does not contain information about cooking pasta. The context focuses on waste classification, composting, and the functionality of an AI-driven smart bin system. To answer your question about cooking pasta, here's a simple recipe:

To cook pasta:
1. Bring a large pot of salted water to a boil.
2. Add the pasta and stir occasionally to prevent sticking.
3. Cook according to package instructions, usually around 8-12 minutes.
4. Drain the pasta and serve with your preferred sauce.

📚 **Sources:**
• EcoGuide_AI_Project_Documentation.md | AI Waste Classifier / Explainable AI
• EcoGuide_AI_Project_Documentation.md | Smart Bin / End-to-End Workflow
• EcoGuide_Awareness | Awareness / Food Waste & Climate
• General_Plastic_Reference | Plastic Types / PP
• EcoGuide_AI_Project_Documentation.md | AI Waste Classifier / Prediction Pipeline

<sub>🎯 recycle_organic | confidence: 1.00 | margin: 0.99</sub>